# CiteGuard quickstart
Check whether your citations support your research claims. No local installation or paid LLM API key required.

**For your own paper:** run Setup, then Module 2 only. Module 1 is an optional sample demo. Do not select Run all unless you want both modules.

**Privacy:** Colab processes uploaded files on Google's cloud runtime. Use the local CLI for confidential manuscripts. Titles/DOIs are sent to metadata services. Results are screening signals, not final verdicts.


In [ ]:
# Setup — run this cell first
!pip install -q "citeguard[nli] @ git+https://github.com/Anandharajan/citeguard"

## Module 1A (optional): single-claim demo


In [ ]:
from citeguard.api import get_nli
from citeguard.algorithms.support import support

abstract = """We propose residual routing for wireless mesh networks. Residual routing reduces packet latency
in mesh networks by learning shortcut paths between distant nodes. Experiments on three testbeds confirm lower
latency. We also release our simulator for reproducibility."""

nli = get_nli("cross-encoder")   # downloads a ~180 MB local model the first time
for claim in ["Residual routing reduces packet latency in mesh networks.",
              "Residual routing increases packet latency in mesh networks.",
              "Residual routing improves battery life of phones."]:
    r = support(claim, abstract, nli)
    print(f"{r['label']:>20}  {claim}")

## Module 1B (optional): sample manuscript demo


In [ ]:
tex = r"""The Transformer relies solely on attention mechanisms, dispensing with recurrence and convolutions \cite{vaswani2017}.
Deep residual learning makes very deep networks easier to optimise \cite{he2016}.
Residual networks were shown to be harder to train than plain networks \cite{he2016}.  % deliberately false"""
bib = r"""@inproceedings{vaswani2017, title={Attention Is All You Need}, author={Vaswani, Ashish and Shazeer, Noam}, year={2017}}
@inproceedings{he2016, title={Deep Residual Learning for Image Recognition}, author={He, Kaiming and Zhang, Xiangyu}, year={2016}}"""
open("citeguard_demo.tex", "w").write(tex); open("citeguard_demo.bib", "w").write(bib)
!citeguard check citeguard_demo.tex citeguard_demo.bib --report citeguard_demo_report.html

In [ ]:
from IPython.display import HTML
HTML(open('citeguard_demo_report.html', encoding='utf-8').read())

Expected: the deliberately false claim should be flagged for review. Actual results may vary. CiteGuard checks abstracts, not full papers.

## Module 2: Check your own paper
In Overleaf, download your project source ZIP and extract it. Select one `.tex` manuscript containing the citing text and one `.bib` bibliography. Any filenames are accepted; multi-file projects may need to be combined first. Run the cell below, select both files, and wait for the report. No commands or renaming needed.


In [ ]:
from google.colab import files
from IPython.display import HTML, display
from pathlib import Path
import tempfile
import subprocess

with tempfile.TemporaryDirectory(prefix="citeguard_") as folder:
    work = Path(folder)
    print("Select one .tex manuscript and one .bib bibliography.")
    uploaded = files.upload(target_dir=str(work))
    tex_files = [name for name in uploaded if Path(name).suffix.lower() == ".tex"]
    bib_files = [name for name in uploaded if Path(name).suffix.lower() == ".bib"]
    if len(uploaded) != 2 or len(tex_files) != 1 or len(bib_files) != 1:
        print("Please run this cell again and select exactly one .tex and one .bib file.")
    else:
        # Use fixed internal names, not user-supplied paths.
        tex = work / "manuscript.tex"
        bib = work / "references.bib"
        tex.write_bytes(uploaded[tex_files[0]])
        bib.write_bytes(uploaded[bib_files[0]])
        report = work / "citeguard_report.html"
        print("Checking citations. First run may download the AI model...")
        result = subprocess.run(["citeguard", "check", str(tex), str(bib), "--report", str(report)], cwd=work)
        if result.returncode == 0 and report.is_file():
            display(HTML(report.read_text(encoding="utf-8")))
            # Keep the report available after the temporary input directory is removed.
            output = Path("/content/citeguard_report.html")
            output.write_bytes(report.read_bytes())
            print("Report ready. Download will start automatically.")
            files.download(str(output))
        else:
            print("Verification did not complete. Review the output above and try again.")


Your report is also available in the Colab Files panel as `citeguard_report.html`. Delete the report or disconnect and delete the runtime when finished. [Star CiteGuard](https://github.com/Anandharajan/citeguard) if it helps, or [report an issue](https://github.com/Anandharajan/citeguard/issues).
